In [20]:
# Create spark session

from pyspark.sql import SparkSession

spark = (
    SparkSession
    .builder
    .appName("Triggers in Spark Streaming using Kafka")
    .config("spark.streaming.stopGracefullyOnShutdown", True)
    .config('spark.jars.packages', 'org.apache.spark:spark-sql-kafka-0-10_2.12:3.3.0')
    .config("spark.sql.shuffle.partitions", 8)
    .master("local[2]")
    .getOrCreate()
)

spark

In [21]:
# Create the kafka_df to read from kafka

kafka_df = (
    spark
    .readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", "ed-kafka:29092")
    .option("subscribe", "device-data")
    .option("startingOffsets", "earliest")
    .load()
)
kafka_df.printSchema()

root
 |-- key: binary (nullable = true)
 |-- value: binary (nullable = true)
 |-- topic: string (nullable = true)
 |-- partition: integer (nullable = true)
 |-- offset: long (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- timestampType: integer (nullable = true)



In [3]:
# Parse value from binary to string into kafka_json_df
from pyspark.sql.functions import expr

kafka_json_df = kafka_df.withColumn("value", expr("cast(value as string)"))

In [4]:
# Schema for payload

from struct import Struct
from pyspark.sql.types import StructType, StructField, StringType, ArrayType, LongType

json_schema = (
    StructType(
        [StructField("customerId", StringType(), True),
        StructField("data",StructType(
            [StructField("devices",
                        # Array of device objects
                        ArrayType(StructType([
                            StructField("deviceId", StringType(), True),
                            StructField("measure", StringType(), True),
                            StructField("status", StringType(), True),
                            StructField("temperature", LongType(), True),
                        ]), True), True)
                ]), True),
        StructField("eventID", StringType(), True),
        StructField("eventOffset", LongType(), True),
        StructField("eventPublisher", StringType(), True),
        StructField("eventTime", StringType(), True)
        ])
)

In [5]:
# Apply the schema to payload to read the data
from pyspark.sql.functions import from_json, col

streaming_df = kafka_json_df.withColumn("values_json", from_json(col("value"), json_schema)).selectExpr("values_json.*")

In [6]:
# lets explode the data as devices contains list/array of device reading

from pyspark.sql.functions import explode

exploded_df = streaming_df.withColumn("data_devices", explode("data.devices")).drop("data")

streaming_df.printSchema()
# streaming_df.show()

root
 |-- customerId: string (nullable = true)
 |-- data: struct (nullable = true)
 |    |-- devices: array (nullable = true)
 |    |    |-- element: struct (containsNull = true)
 |    |    |    |-- deviceId: string (nullable = true)
 |    |    |    |-- measure: string (nullable = true)
 |    |    |    |-- status: string (nullable = true)
 |    |    |    |-- temperature: long (nullable = true)
 |-- eventID: string (nullable = true)
 |-- eventOffset: long (nullable = true)
 |-- eventPublisher: string (nullable = true)
 |-- eventTime: string (nullable = true)



In [ ]:
# Flatten the exploded df
from pyspark.sql.functions import col
flattened_df = (
    exploded_df
    .withColumn("deviceID", col("data_devices.deviceId"))
    .withColumn("measure", col("data_devices.measure"))
    .withColumn("status", col("data_devices.status"))
    .withColumn("temperature", col("data_devices.temperature"))
    .drop("data_devices")
)
flattened_df.printSchema()
# flattened_df.show()

In [ ]:
# # Running in once/availableNow and processingTime mode
# # Write the output to console sink to check the output

(flattened_df
 .writeStream
 .format("console")
 .outputMode("append")
 .trigger(once=True)
 .option("checkpointLocation", "checkpoint_dir_kafka_1")
 .start()
 .awaitTermination())

In [ ]:
(flattened_df
 .writeStream
 .format("console")
 .outputMode("append")
 .trigger(processingTime='10 seconds')
 .option("checkpointLocation", "checkpoint_dir_kafka_1")
 .start()
 .awaitTermination())

In [22]:
# # Running in continous mode
# # Write the output to console sink to check the output

 # cannot use explode and its experimental
(kafka_df
 .writeStream
 .format("memory")
 .queryName("kafka_table3")
 .outputMode("append")
 .trigger(continuous='10 seconds')
 .option("checkpointLocation", "checkpoint_dir_kafka_3")
 .start()
 .awaitTermination())

ERROR:root:KeyboardInterrupt while sending command.
Traceback (most recent call last):
  File "/spark/python/lib/py4j-0.10.9.5-src.zip/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
  File "/spark/python/lib/py4j-0.10.9.5-src.zip/py4j/clientserver.py", line 511, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
  File "/usr/local/lib/python3.10/socket.py", line 717, in readinto
    return self._sock.recv_into(b)
KeyboardInterrupt


KeyboardInterrupt: 

In [23]:
# View data form Memory sink
spark.sql("select * from kafka_table3").show()

+---+-----+-----+---------+------+---------+-------------+
|key|value|topic|partition|offset|timestamp|timestampType|
+---+-----+-----+---------+------+---------+-------------+
+---+-----+-----+---------+------+---------+-------------+

